# Validação de Modelo — Precisão / Recall

Este notebook mede **o quão bem um modelo de classificação se sai** em gravações que você já
anotou. Ele lê áudio e anotações do seu **Google Drive**, executa o modelo
**uma única vez** e então varia as configurações de detecção para traçar **curvas de precisão e
recall**.

---

### Por que uma variação de parâmetros?

Um modelo não tem uma única acurácia — ele tem uma acurácia *por ponto de operação*. Aumente o
limiar de detecção e a precisão sobe enquanto o recall cai. O objetivo deste notebook é mostrar
essa curva de compromisso inteira, para que você escolha o limiar que o seu projeto realmente
precisa.

O modelo é executado **uma única vez** e suas saídas (**logits**) são armazenadas. Cada limiar
e cada bias (deslocamento) do sigmoide são então aplicados sobre esses logits armazenados, o que custa
milissegundos em vez de reexecutar a inferência. É isso que torna prático variar dezenas de pontos
de operação.

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para ler áudio, anotações e salvar resultados
2. **Instala os programas necessários** automaticamente
3. **Acessa sua pasta de áudio e suas anotações** e faz o pareamento entre elas
4. **Carrega um modelo** do HuggingFace ou do seu Google Drive (ex.: BirdNET, Perch, customizado)
5. **Executa o modelo uma vez por gravação** e armazena os logits brutos
6. **Varia limiares × bias do sigmoide**, comparando as detecções com as anotações
7. **Traça curvas de precisão e recall** — uma figura por rótulo, um par de curvas colorido por bias

### Antes de começar:
- Você precisa de uma **conta Google** com Google Drive
- Você precisa de um **arquivo de modelo TFLite (`.tflite`) ou ONNX (`.onnx`)** e do
  **arquivo de rótulos** correspondente
- Você precisa de **gravações anotadas** — tabelas de seleção do Raven, faixas de rótulos do
  Audacity, ou um único CSV
- Áudio e anotações precisam estar no seu Google Drive

### Como executar:
Execute cada célula **uma de cada vez**, de cima para baixo. Clique no botão ▶ à esquerda de cada
célula, ou pressione `Shift + Enter`.

> **Novo em notebooks?** Uma célula com `[ ]` à esquerda ainda não foi executada. Após executar,
> ela mostra `[1]`, `[2]`, etc. Se aparecer um erro (texto vermelho), leia a mensagem — geralmente
> ela diz exatamente o que corrigir.

---

Criado por [biodiversica](https://biodiversica.xyz). Para problemas, dúvidas ou sugestões, abra uma
issue no [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) ou visite
[biodiversica.xyz](https://biodiversica.xyz).

---
## Etapa 1 — Conectar o Google Drive e Instalar os Programas

Execute as duas células abaixo. A primeira pedirá que você **autorize o acesso ao seu Google
Drive** — clique no link e siga os passos.

In [ ]:
#@title 📂 Conectar o Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso.')

In [ ]:
#@title 📦 Instalar pacotes { display-mode: "form" }

#@markdown **Dispositivo de processamento** — onde o modelo será executado durante a inferência.
#@markdown - **CPU**: funciona em qualquer ambiente do Colab (mais lento).
#@markdown - **GPU**: muito mais rápido para modelos **ONNX**, mas você precisa primeiro trocar o
#@markdown   ambiente de execução para GPU (*Ambiente de execução → Alterar o tipo de ambiente de
#@markdown   execução → T4 GPU*, e então reexecutar desde o início).
#@markdown   Modelos TFLite sempre rodam na CPU, independentemente desta configuração.
COMPUTE_DEVICE = 'CPU'  #@param ["CPU", "GPU"]

!pip install ai-edge-litert librosa soundfile huggingface_hub -q

# Instala a versão correspondente do ONNX Runtime. onnxruntime (CPU) e
# onnxruntime-gpu não podem coexistir, então removemos um antes de instalar o outro.
if COMPUTE_DEVICE == 'GPU':
    !pip uninstall -y onnxruntime onnxruntime-gpu -q
    # O Colab vem com CUDA 12, mas os wheels mais recentes do onnxruntime-gpu são
    # compilados para CUDA 13 (procuram por libcudart.so.13). Fixamos na última
    # versão compilada para CUDA 12.
    !pip install "onnxruntime-gpu==1.22.0" -q
else:
    !pip uninstall -y onnxruntime-gpu -q
    !pip install onnxruntime -q

print(f'\nTodos os pacotes foram instalados com sucesso (dispositivo: {COMPUTE_DEVICE}).')

---
## Etapa 2 — Configuração

Preencha os formulários abaixo. **Você não precisa editar nenhum código** — apenas digite ou
selecione seus valores em cada formulário e execute a célula.

Execute todos os formulários de cima para baixo:
1. **Configurações Gerais** — onde os resultados são gravados, cache de logits
2. **Pré-processamento de Áudio** — filtragem / mudança de velocidade opcionais
3. **Entrada de Áudio** — onde estão suas gravações anotadas
4. **Anotações** — onde estão suas anotações de referência, como estão formatadas, **quais rótulos
   validar**, e como **traduzir** os nomes dos rótulos entre o seu conjunto de dados e o modelo
5. **Modelo** — onde o modelo está armazenado e como ele funciona
6. **Variação de parâmetros e Validação** — os limiares e bias a testar, e como uma detecção é associada a uma anotação

> **Dica:** Os formulários ocultam o código automaticamente. Para ver o código por trás, clique no
> ícone `{ }` no canto superior direito de qualquer célula de formulário.

In [ ]:
#@title ⚙️ Configurações Gerais { display-mode: "form" }

import os

#@markdown **Pasta de resultados** — onde as métricas de validação e as figuras serão salvas no seu Drive.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/validation_results"  #@param {type:"string"}

#@markdown ---
#@markdown **Guardar os logits do modelo em cache no Drive** — armazena as saídas brutas do modelo
#@markdown para cada gravação, de modo que reexecutar a variação de parâmetros (com outros limiares ou bias)
#@markdown não exija reexecutar o modelo. O cache é invalidado automaticamente se o modelo, as
#@markdown configurações de áudio ou o conjunto de rótulos avaliados mudarem.
USE_LOGITS_CACHE = True  #@param {type:"boolean"}

#@markdown **Pasta do cache** — usada apenas quando o cache está ativado.
DRIVE_LOGITS_CACHE_DIR = "/content/drive/MyDrive/validation_results/logits_cache"  #@param {type:"string"}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
if USE_LOGITS_CACHE:
    os.makedirs(DRIVE_LOGITS_CACHE_DIR, exist_ok=True)

print(f"Pasta de resultados : {DRIVE_RESULTS_DIR}")
print(f"Cache de logits     : {DRIVE_LOGITS_CACHE_DIR if USE_LOGITS_CACHE else 'desativado'}")

In [ ]:
#@title ⚙️ Pré-processamento de Áudio { display-mode: "form" }
#@markdown Pré-processamento opcional aplicado em cada gravação antes da inferência.
#@markdown Use as **mesmas configurações que você pretende usar na prática** — caso contrário, os
#@markdown números produzidos por este notebook não descreverão o seu fluxo real.
#@markdown
#@markdown ---
#@markdown **Filtro de frequência** — remove frequências fora da faixa de interesse.
FILTER_TYPE = "none"  #@param ["none", "lowpass", "highpass", "bandpass"]

#@markdown **Frequência de corte inferior (Hz)** — usada nos filtros passa-alta e passa-banda.
FILTER_LOW_HZ = 0  #@param {type:"integer"}

#@markdown **Frequência de corte superior (Hz)** — usada nos filtros passa-baixa e passa-banda.
FILTER_HIGH_HZ = 15000  #@param {type:"integer"}

#@markdown ---
#@markdown **Velocidade de reprodução** — 1.0 = normal. Abaixo de 1.0 desacelera e alonga o áudio;
#@markdown acima de 1.0 acelera e encurta. Útil para deslocar o conteúdo de frequência para a faixa
#@markdown esperada pelo modelo (ex.: 0.5x divide todas as frequências pela metade).
#@markdown Os tempos das anotações são sempre interpretados na linha do tempo da gravação
#@markdown **original** — o notebook converte os tempos das janelas de volta antes de comparar.
AUDIO_SPEED = 1.0  #@param {type:"number"}
AUDIO_SPEED = min(max(AUDIO_SPEED, 0.25), 4.0)  # mantém entre 0.25–4.0

_preprocess_lines = []
if FILTER_TYPE != 'none':
    if FILTER_TYPE == 'lowpass':
        _preprocess_lines.append(f'Filtro     : passa-baixa <= {FILTER_HIGH_HZ} Hz')
    elif FILTER_TYPE == 'highpass':
        _preprocess_lines.append(f'Filtro     : passa-alta >= {FILTER_LOW_HZ} Hz')
    elif FILTER_TYPE == 'bandpass':
        _preprocess_lines.append(f'Filtro     : passa-banda {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz')
if AUDIO_SPEED != 1.0:
    _preprocess_lines.append(f'Velocidade : {AUDIO_SPEED}x')
if _preprocess_lines:
    print('Pré-processamento ativado:')
    for _l in _preprocess_lines:
        print(f'  {_l}')
else:
    print('Pré-processamento : nenhum')

In [ ]:
#@title 🗂️ Entrada de Áudio { display-mode: "form" }

#@markdown **Pasta de áudio** — caminho para a pasta no seu Google Drive que contém as gravações
#@markdown **anotadas**. Subpastas também são percorridas, e cada gravação é identificada pelo seu
#@markdown caminho **dentro desta pasta** — então duas gravações em subpastas diferentes podem ter
#@markdown o mesmo nome de arquivo sem problema.
#@markdown Exemplo: `/content/drive/MyDrive/meu_projeto/audio_validacao`
DRIVE_INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

if not os.path.isdir(DRIVE_INPUT_DIR):
    print(f"ATENÇÃO: Pasta não encontrada: {DRIVE_INPUT_DIR}")
    print("Verifique o caminho acima — confirme que o Google Drive está conectado e que a pasta existe.")
else:
    _found = [os.path.join(root, name)
              for root, _, names in os.walk(DRIVE_INPUT_DIR) for name in names
              if name.lower().endswith(('.wav', '.flac', '.mp3'))]
    print(f"Pasta de áudio    : {DRIVE_INPUT_DIR}")
    print(f"Arquivos de áudio : {len(_found)}")
    for _path in sorted(_found)[:5]:
        print(f"  {os.path.relpath(_path, DRIVE_INPUT_DIR)}")
    if len(_found) > 5:
        print(f"  ... e mais {len(_found) - 5}")

In [ ]:
#@title 📝 Anotações { display-mode: "form" }

#@markdown **Formato das anotações** — como suas anotações de referência estão armazenadas.
#@markdown - `raven` — uma **tabela de seleção do Raven Pro** por gravação (separada por tabulação, com cabeçalho).
#@markdown - `audacity` — uma **faixa de rótulos do Audacity** por gravação: `início<TAB>fim<TAB>rótulo`, sem cabeçalho.
#@markdown - `csv` — uma **única tabela para todo o conjunto de dados**, uma linha por anotação, com
#@markdown   uma coluna indicando o arquivo de áudio.
ANNOTATION_FORMAT = "raven"  #@param ["raven", "audacity", "csv"]

#@markdown ---
#@markdown **Pasta de anotações** (`raven` / `audacity`) — os arquivos de anotação, um por gravação.
#@markdown Um arquivo é associado a uma gravação quando seu nome **começa com o nome do arquivo de
#@markdown áudio** (sem a extensão), ex.: `20250615_203000.wav` ↔ `20250615_203000.Table.1.selections.txt`.
DRIVE_ANNOTATION_DIR = "/content/drive/MyDrive/annotations"  #@param {type:"string"}

#@markdown **Tabela de anotações** (`csv` apenas) — caminho para o arquivo único de anotações (`.csv` ou `.txt`).
DRIVE_ANNOTATION_CSV = "/content/drive/MyDrive/annotations/annotations.csv"  #@param {type:"string"}

#@markdown ---
#@markdown ### Nomes das colunas (usados por `raven` e `csv`, ignorados por `audacity`)
#@markdown Os padrões correspondem a uma tabela de seleção padrão do Raven Pro.
ANN_START_COLUMN = "Begin Time (s)"  #@param {type:"string"}
ANN_END_COLUMN   = "End Time (s)"    #@param {type:"string"}
ANN_LABEL_COLUMN = "Annotation"      #@param {type:"string"}
#@markdown **Coluna do arquivo** (`csv` apenas) — a coluna que contém o nome do arquivo de áudio.
ANN_FILE_COLUMN  = "filename"        #@param {type:"string"}

#@markdown ---
#@markdown ### 🏷️ Tradução de rótulos
#@markdown Suas anotações e seu modelo raramente escrevem o mesmo rótulo da mesma forma. Estes dois
#@markdown campos renomeiam cada lado para **um vocabulário comum** — traduza o lado que for mais
#@markdown fácil, ou os dois. Formato: pares `de=para` separados por ponto e vírgula.
#@markdown
#@markdown Execute a **Etapa 3** para ver os nomes dos rótulos nas suas anotações, e a **Etapa 4**
#@markdown para que o notebook aponte quais deles não correspondem a nenhum rótulo do modelo, com
#@markdown sugestões.

#@markdown **Traduzir rótulos das anotações** — renomeia um rótulo usado no seu conjunto de dados.
#@markdown Exemplo: `PHYLUT=Phyllodytes luteolus;DENMIN=Dendropsophus minutus`
TRANSLATE_ANNOTATION_LABELS = ""  #@param {type:"string"}

#@markdown **Traduzir rótulos do modelo** — renomeia um rótulo produzido pelo modelo.
#@markdown Exemplo: `Phyllodytes luteolus_Bromeliad Treefrog=Phyllodytes luteolus`
#@markdown Quando vários rótulos do modelo são traduzidos para o mesmo nome, vale o de maior pontuação.
TRANSLATE_MODEL_LABELS = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### 🎯 Rótulos a validar
#@markdown **Rótulos a validar** — separados por ponto e vírgula. Deixe **em branco para validar
#@markdown todos os rótulos encontrados nas suas anotações** (o padrão). Execute a Etapa 3 primeiro
#@markdown para ver essa lista, e então volte aqui e reduza a seleção se só alguns rótulos
#@markdown interessam.
#@markdown
#@markdown Os nomes são comparados **depois** das traduções acima. Você também pode listar um rótulo
#@markdown que nunca aparece nas suas anotações: ele será avaliado **apenas por falsos positivos**,
#@markdown que é a forma de cobrar do modelo a detecção de um rótulo que não está ali.
#@markdown Exemplo: `Phyllodytes luteolus;Dendropsophus minutus`
VALIDATION_LABELS = ""  #@param {type:"string"}

#@markdown **Ignorar estes rótulos das anotações** — separados por ponto e vírgula. Linhas com um
#@markdown destes rótulos são descartadas antes de tudo, então nunca chegam à lista acima.
#@markdown Exemplo: `Desconhecido;Ruído;?`
IGNORE_ANNOTATION_LABELS = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Incluir gravações sem arquivo de anotação** — quando ativado, essas gravações são
#@markdown tratadas como **totalmente negativas** (toda detecção nelas é um falso positivo).
#@markdown Só ative isso se quem anotou realmente ouviu esses arquivos e não encontrou nada.
INCLUDE_UNANNOTATED_FILES = False  #@param {type:"boolean"}


def parse_pair_map(text):
    """Converte um campo 'de=para;de=para' do formulário em um dicionário."""
    mapping = {}
    for entry in (text or '').split(';'):
        entry = entry.strip()
        if '=' not in entry:
            continue
        source, target = entry.split('=', 1)
        source, target = source.strip(), target.strip()
        if source and target:
            mapping[source] = target
    return mapping


def parse_list(text):
    """Converte um campo 'a;b;c' do formulário em uma lista de textos não vazios."""
    return [item.strip() for item in (text or '').split(';') if item.strip()]


ANNOTATION_LABEL_MAP_D = parse_pair_map(TRANSLATE_ANNOTATION_LABELS)
MODEL_LABEL_MAP_D      = parse_pair_map(TRANSLATE_MODEL_LABELS)
IGNORED_LABELS         = set(parse_list(IGNORE_ANNOTATION_LABELS))
SELECTED_LABELS        = parse_list(VALIDATION_LABELS)

print(f"Formato               : {ANNOTATION_FORMAT}")
if ANNOTATION_FORMAT == 'csv':
    print(f"Tabela de anotações   : {DRIVE_ANNOTATION_CSV}")
    print(f"Colunas               : arquivo='{ANN_FILE_COLUMN}'  início='{ANN_START_COLUMN}'  "
          f"fim='{ANN_END_COLUMN}'  rótulo='{ANN_LABEL_COLUMN}'")
else:
    print(f"Pasta de anotações    : {DRIVE_ANNOTATION_DIR}")
    if ANNOTATION_FORMAT == 'raven':
        print(f"Colunas               : início='{ANN_START_COLUMN}'  fim='{ANN_END_COLUMN}'  "
              f"rótulo='{ANN_LABEL_COLUMN}'")
print(f"Rótulos ignorados     : {sorted(IGNORED_LABELS) or 'nenhum'}")
print(f"Gravações sem anotação: {'tratadas como totalmente negativas' if INCLUDE_UNANNOTATED_FILES else 'ignoradas'}")
print()
if ANNOTATION_LABEL_MAP_D or MODEL_LABEL_MAP_D:
    print("Tradução de rótulos:")
    for source, target in sorted(ANNOTATION_LABEL_MAP_D.items()):
        print(f"  anotação  {source!r} → {target!r}")
    for source, target in sorted(MODEL_LABEL_MAP_D.items()):
        print(f"  modelo    {source!r} → {target!r}")
else:
    print("Tradução de rótulos : nenhuma — rótulos das anotações e do modelo são comparados como estão.")
print()
if SELECTED_LABELS:
    print(f"Rótulos a validar ({len(SELECTED_LABELS)}):")
    for label in SELECTED_LABELS:
        print(f"  {label}")
else:
    print("Rótulos a validar : todos os encontrados nas anotações (a Etapa 3 lista quais são).")

In [ ]:
#@title 🤖 Modelo { display-mode: "form" }

#@markdown **Acesso ao modelo** — onde está armazenado o arquivo do modelo?
MODEL_SOURCE = "google_drive"  #@param ["huggingface", "google_drive"]

#@markdown ---
#@markdown Caminho completo para o arquivo do modelo no seu Drive (`.tflite` ou `.onnx`).
DRIVE_MODEL_PATH  = "/content/drive/MyDrive/Models/model.tflite"  #@param {type:"string"}
#@markdown Caminho completo para o arquivo de rótulos no seu Drive.
DRIVE_LABELS_PATH = "/content/drive/MyDrive/Models/labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown O ID do repositório é a parte após `huggingface.co/` na URL do modelo.
#@markdown Padrão: `justinchuby/BirdNET-onnx` (BirdNET v2.4 em formato ONNX)
HF_REPO_ID     = "justinchuby/BirdNET-onnx"  #@param {type:"string"}
HF_MODEL_FILE  = "model.onnx"               #@param {type:"string"}
#@markdown O arquivo de rótulos pode estar em um repositório HuggingFace **diferente**. Deixe em branco para usar o mesmo repositório do modelo.
HF_LABELS_REPO = ""                          #@param {type:"string"}
HF_LABELS_FILE = "BirdNET_GLOBAL_6K_V2.4_Labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown ### Configurações do arquivo de rótulos
#@markdown **Possui linha de cabeçalho?** — marque se a primeira linha do arquivo de rótulos é um cabeçalho de coluna (não um rótulo).
LABELS_HAS_HEADER = False  #@param {type:"boolean"}
#@markdown **Índice da coluna de rótulo** — qual coluna contém o nome do rótulo? (0 = primeira coluna, 1 = segunda, etc.)
LABELS_COLUMN_INDEX = 0  #@param {type:"integer"}
#@markdown **Delimitador de coluna** — como as colunas são separadas no arquivo de rótulos.
LABELS_DELIMITER = "tab"  #@param ["tab", "comma (,)", "semicolon (;)", "underscore (_)"]

#@markdown ---
#@markdown **Sensibilidade do sigmoide** — inclinação da curva sigmoide. `-1.0` = sigmoide padrão;
#@markdown mais negativo = mais íngreme. Precisa continuar **negativa**, para que um logit maior
#@markdown signifique sempre uma pontuação maior. Este notebook sempre ativa com um sigmoide — o
#@markdown **bias** é um dos parâmetros a serem variados, e por isso é configurado mais abaixo.
SIGMOID_SENSITIVITY = -1.0  #@param {type:"number"}
SIGMOID_SENSITIVITY = min(SIGMOID_SENSITIVITY, -0.01)  # mantém estritamente negativo

#@markdown **Taxa de amostragem (Hz)** — taxa de amostragem de áudio esperada pelo modelo.
#@markdown BirdNET = 48000 · Google Perch = 32000 · Customizado: consulte a documentação do modelo.
SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Duração do segmento (segundos)** — comprimento de cada trecho de áudio enviado ao modelo.
#@markdown BirdNET = 3.0 · Google Perch = 5.0 · Customizado: consulte a documentação do modelo.
SEGMENT_DURATION_S = 3.0  #@param {type:"number"}

#@markdown **Sobreposição entre segmentos (0.0–0.9)** — fração de sobreposição entre trechos consecutivos.
SEGMENT_OVERLAP = 0.0  #@param {type:"number"}
SEGMENT_OVERLAP = min(max(SEGMENT_OVERLAP, 0.0), 0.9)  # mantém entre 0.0–0.9

print(f"Acesso ao modelo        : {MODEL_SOURCE}")
print(f"Ativação                : sigmoide (sensibilidade={SIGMOID_SENSITIVITY}, o bias é varrido)")
print(f"Taxa de amostragem      : {SAMPLE_RATE} Hz")
print(f"Duração do segmento     : {SEGMENT_DURATION_S} s")
print(f"Sobreposição            : {SEGMENT_OVERLAP}")
print(f"Rótulos: índice da coluna={LABELS_COLUMN_INDEX}, delimitador='{LABELS_DELIMITER}', cabeçalho={LABELS_HAS_HEADER}")

In [ ]:
#@title 🎚️ Variação de parâmetros e Validação { display-mode: "form" }

#@markdown ### Pontos de operação a testar
#@markdown **Limiares de pontuação** — separados por vírgula. Este é o **eixo x** dos gráficos
#@markdown finais: uma detecção é mantida quando sua pontuação ativada é maior ou igual ao limiar.
SCORE_THRESHOLDS = "0.05,0.1,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95"  #@param {type:"string"}

#@markdown **bias do sigmoide** — separados por vírgula. Cada valor vira **um par de curvas
#@markdown colorido** (precisão e recall) em cada figura. `1.0` = sigmoide padrão; acima de 1.0
#@markdown desloca a curva para a esquerda (pontuações maiores, mais sensível); abaixo de 1.0
#@markdown desloca para a direita (mais conservador).
SIGMOID_BIASES = "0.75,1.0,1.25"  #@param {type:"string"}

#@markdown ---
#@markdown ### Como uma detecção é associada a uma anotação
#@markdown **Modo de correspondência**
#@markdown - `overlap` — qualquer interseção temporal conta como correspondência (padrão).
#@markdown - `iou` — exige interseção sobre união maior ou igual ao limiar abaixo.
MATCH_MODE = "overlap"  #@param ["overlap", "iou"]
#@markdown **Limiar de IoU** — usado apenas quando o modo de correspondência é `iou`. Precisa estar entre 0 e 1.
IOU_THRESHOLD = 0.5  #@param {type:"number"}

#@markdown ---
#@markdown **Granularidade** — o que cada TP / FP / FN conta.
#@markdown - `window` — uma contagem por **janela de análise do modelo** × rótulo. Uma anotação longa
#@markdown   que cobre muitas janelas produz muitos TPs, então o resultado é ponderado pela duração
#@markdown   da anotação.
#@markdown - `annotation` — uma contagem por **evento anotado**. Uma sequência de detecções dentro de
#@markdown   um mesmo canto anotado vira um único TP, então o recall não é inflado pela duração do
#@markdown   canto. Detecções sem anotação correspondente continuam contando como FP. **Recomendado.**
#@markdown - `file` — uma contagem por **gravação** × rótulo: o rótulo estava presente em algum ponto
#@markdown   do arquivo, e o modelo o encontrou em algum ponto? Os tempos são ignorados.
GRANULARITY = "annotation"  #@param ["annotation", "window", "file"]

#@markdown ---
#@markdown **Reportar verdadeiros negativos** — só faz sentido na granularidade `file`, onde um TN é
#@markdown uma gravação em que um rótulo não foi anotado nem detectado. TNs não afetam a precisão
#@markdown nem o recall.
REPORT_TN = False  #@param {type:"boolean"}


def parse_float_list(text, lo, hi):
    """Converte um campo separado por vírgulas em floats únicos, ordenados e limitados."""
    values = []
    for item in (text or '').replace(';', ',').split(','):
        item = item.strip()
        if not item:
            continue
        try:
            values.append(min(max(float(item), lo), hi))
        except ValueError:
            print(f"  ATENÇÃO: não foi possível ler '{item}' como número — ignorando.")
    return sorted(set(values))


THRESHOLDS = parse_float_list(SCORE_THRESHOLDS, 0.0, 1.0)
BIASES     = parse_float_list(SIGMOID_BIASES, 0.01, 1.99)

if not THRESHOLDS:
    raise ValueError("Nenhum limiar de pontuação utilizável. Informe ao menos um número entre 0 e 1.")
if not BIASES:
    raise ValueError("Nenhum bias de sigmoide utilizável. Informe ao menos um número entre 0.01 e 1.99.")
if MATCH_MODE == 'iou' and not (0 < IOU_THRESHOLD < 1):
    raise ValueError(f"IOU_THRESHOLD precisa estar entre 0 e 1 (exclusivo), recebido {IOU_THRESHOLD}.")

print(f"Limiares de pontuação : {THRESHOLDS}")
print(f"bias do sigmoide    : {BIASES}")
print(f"Pontos de operação    : {len(THRESHOLDS) * len(BIASES)}  "
      f"({len(BIASES)} par(es) de curvas × {len(THRESHOLDS)} ponto(s))")
print(f"Modo de correspondência: {MATCH_MODE}" + (f" (IoU >= {IOU_THRESHOLD})" if MATCH_MODE == 'iou' else ""))
print(f"Granularidade         : {GRANULARITY}")

# Com o sigmoide padrão, pontuação >= t é o mesmo corte que
# logit >= ln(t / (1 - t)) - 10 * (bias - 1): o limiar e o bias movem o mesmo
# ponto de operação ao longo de um único eixo. Varrer os dois portanto remede
# pontos que você já tem — as curvas de bias são deslocamentos horizontais umas
# das outras. É exatamente isso que as torna comparáveis, mas vale saber que uma
# curva de bias não é informação nova sobre o modelo, apenas um eixo x renomeado.
if len(BIASES) > 1:
    print()
    print("Nota: com um sigmoide, o limiar de pontuação e o bias movem o mesmo ponto de operação.")
    print("      As curvas de bias são deslocamentos horizontais umas das outras — úteis para ler")
    print("      qual limiar seria necessário em cada bias, não informação extra sobre o modelo.")

---
## Etapa 3 — Varrer os arquivos de áudio e carregar as anotações

Esta célula encontra suas gravações, lê as anotações, faz o pareamento entre elas e informa o que
será avaliado.

**Como uma gravação é associada às suas anotações** — uma gravação é identificada pelo seu caminho
*dentro da pasta de áudio*, e não apenas pelo nome do arquivo, então `PONTO_A/20250615_203000.wav`
e `PONTO_B/20250615_203000.wav` são duas gravações diferentes:

- `raven` / `audacity` — o arquivo de anotação cujo nome começa com o nome do arquivo da gravação.
  Quando a pasta de áudio tem subpastas, vence o arquivo de anotação **na subpasta correspondente**;
  se a pasta de anotações não tiver subpastas, usa-se um nome de arquivo único em toda a pasta.
- `csv` — a coluna de arquivo de uma linha pode conter o caminho completo dentro da pasta de áudio
  (`PONTO_A/20250615_203000.wav`) ou apenas o nome do arquivo, desde que esse nome seja único na tabela.

Se o nome de um arquivo se repete em várias subpastas e nada os distingue, a célula **pula essa
gravação e avisa**, em vez de misturar dois conjuntos de anotações de referência.

**O que conferir na saída:**
- **Gravações a validar** — se for 0, seus arquivos de anotação não estão sendo associados. Confira
  se os nomes dos arquivos de anotação começam com os nomes dos arquivos de áudio.
- **Rótulos encontrados nas anotações** — a lista completa dos nomes de rótulos usados no seu
  conjunto de dados, com a contagem de cada um. Este é o cardápio: deixe `VALIDATION_LABELS` em
  branco para validar todos, ou copie os que interessam para esse campo e execute a célula de novo.
- **Avisos de tradução de rótulos** — uma tradução cujo lado esquerdo nunca correspondeu a nenhuma
  anotação é reportada aqui, já que um erro de digitação ali se parece exatamente com um rótulo
  ausente.

A Etapa 4 então confere cada rótulo contra o vocabulário do próprio modelo e sugere traduções para
os que não baterem.

In [ ]:
#@title 🔍 Variação de parâmetros { display-mode: "form" }

import csv
import difflib
import glob as _glob
from collections import defaultdict

AUDIO_EXTENSIONS = ('.wav', '.flac', '.mp3')


def close_names(name, candidates, n=3, cutoff=0.6):
    """Nomes candidatos mais próximos de `name`, comparados sem diferenciar maiúsculas.

    Um rótulo que difere apenas na capitalização é a causa mais comum de dois
    vocabulários não baterem, e o difflib pontua esse par como zero — então a
    comparação é feita em minúsculas e a grafia original é devolvida.
    """
    by_lower = {candidate.lower(): candidate for candidate in candidates}
    return [by_lower[match] for match in
            difflib.get_close_matches(name.lower(), list(by_lower), n=n, cutoff=cutoff)]


def recording_key(path, root):
    """A identidade de uma gravação: seu caminho dentro de `root`, sem a extensão.

    Duas gravações em subpastas diferentes podem ter o mesmo nome de arquivo, então
    a subpasta precisa fazer parte da chave — usar só o nome do arquivo acabaria
    associando as anotações de uma subpasta ao áudio de outra.
    """
    return os.path.splitext(os.path.relpath(path, root))[0].replace(os.sep, '/')


def name_matches(basename, stem):
    """O nome deste arquivo de anotação pertence à gravação com este radical?"""
    return (basename == stem
            or basename.startswith(stem + '.')
            or basename.startswith(stem + '_'))


def sniff_delimiter(sample_line):
    """Descobre o delimitador de uma tabela: tabulação, depois ponto e vírgula, depois vírgula."""
    for delimiter in ('\t', ';', ','):
        if delimiter in sample_line:
            return delimiter
    return ','


def read_table(path):
    """Lê uma tabela de texto delimitada com cabeçalho para uma lista de dicionários."""
    with open(path, 'r', encoding='utf-8-sig', newline='') as handle:
        first = handle.readline()
        if not first:
            return []
        handle.seek(0)
        return list(csv.DictReader(handle, delimiter=sniff_delimiter(first)))


# Cada rótulo como está escrito nos arquivos, antes da tradução. Guardado para que
# a variação de parâmetros possa reportar uma tradução cujo lado esquerdo nunca correspondeu a
# nada — um erro de digitação silencioso que, de outra forma, pareceria apenas um
# rótulo ausente.
raw_label_counts = defaultdict(int)


def clean_annotation(start, end, label):
    """Normaliza uma anotação bruta, ou devolve None se ela não puder ser usada."""
    try:
        start, end = float(start), float(end)
    except (TypeError, ValueError):
        return None
    if end <= start:
        return None
    label = (label or '').strip()
    if not label:
        return None
    raw_label_counts[label] += 1
    if label in IGNORED_LABELS:
        return None
    # A tradução é aplicada aqui, uma vez, para que tudo daqui em diante veja um só vocabulário.
    return {'start_time': start, 'end_time': end,
            'label': ANNOTATION_LABEL_MAP_D.get(label, label)}


def read_raven_table(path):
    """Lê uma tabela de seleção do Raven Pro."""
    rows = read_table(path)
    if not rows:
        return []
    columns = rows[0].keys()
    for required in (ANN_START_COLUMN, ANN_END_COLUMN, ANN_LABEL_COLUMN):
        if required not in columns:
            raise KeyError(
                f"Coluna '{required}' não encontrada em {os.path.basename(path)}.\n"
                f"Colunas presentes: {list(columns)}\n"
                "Corrija os nomes das colunas no formulário de Anotações (Etapa 2)."
            )
    annotations = []
    for row in rows:
        # O Raven grava uma linha por visualização quando tanto Waveform quanto
        # Spectrogram são exportados; manter as duas dobraria todas as contagens.
        view = (row.get('View') or '').strip()
        if view and not view.lower().startswith('spectrogram'):
            continue
        annotation = clean_annotation(row.get(ANN_START_COLUMN), row.get(ANN_END_COLUMN),
                                      row.get(ANN_LABEL_COLUMN))
        if annotation:
            annotations.append(annotation)
    return annotations


def read_audacity_labels(path):
    """Lê uma faixa de rótulos do Audacity: início<TAB>fim<TAB>rótulo, sem cabeçalho."""
    annotations = []
    with open(path, 'r', encoding='utf-8-sig') as handle:
        for line in handle:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            annotation = clean_annotation(parts[0], parts[1], parts[2])
            if annotation:
                annotations.append(annotation)
    return annotations


def read_annotation_csv(path):
    """Lê uma tabela única cobrindo todo o conjunto, indexada pelo nome do arquivo de áudio.

    Devolve (by_key, aliases). `by_key` é indexado pela coluna de arquivo sem a
    extensão, exatamente como foi escrita — então `PONTO_A/rec.wav` vira
    `PONTO_A/rec`. `aliases` mapeia um nome de arquivo puro para a única chave que
    o possui, ou para None quando várias chaves o compartilham: um nome usado por
    duas gravações diferentes não pode ser resolvido, e adivinhar misturaria
    silenciosamente as anotações de referência das duas.
    """
    rows = read_table(path)
    if not rows:
        return {}, {}
    columns = rows[0].keys()
    for required in (ANN_FILE_COLUMN, ANN_START_COLUMN, ANN_END_COLUMN, ANN_LABEL_COLUMN):
        if required not in columns:
            raise KeyError(
                f"Coluna '{required}' não encontrada em {os.path.basename(path)}.\n"
                f"Colunas presentes: {list(columns)}\n"
                "Corrija os nomes das colunas no formulário de Anotações (Etapa 2)."
            )
    by_key = {}
    for row in rows:
        raw = (row.get(ANN_FILE_COLUMN) or '').strip().replace('\\', '/').lstrip('./')
        key = os.path.splitext(raw)[0]
        if not key:
            continue
        # Registra a gravação mesmo quando todas as suas linhas são filtradas:
        # ela ainda assim foi revisada, então é uma gravação totalmente negativa,
        # e não uma gravação com anotações faltando.
        by_key.setdefault(key, [])
        annotation = clean_annotation(row.get(ANN_START_COLUMN), row.get(ANN_END_COLUMN),
                                      row.get(ANN_LABEL_COLUMN))
        if annotation:
            by_key[key].append(annotation)

    aliases = {}
    for key in by_key:
        name = key.rsplit('/', 1)[-1]
        aliases[name] = None if name in aliases else key
    return by_key, aliases


# --- encontra o áudio ---------------------------------------------------------
if not os.path.isdir(DRIVE_INPUT_DIR):
    raise FileNotFoundError(
        f"Pasta de áudio não encontrada: {DRIVE_INPUT_DIR}\n"
        "Confira o caminho no formulário de Entrada de Áudio (Etapa 2)."
    )

all_audio = sorted(
    path for extension in AUDIO_EXTENSIONS
    for path in _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', f'*{extension}'), recursive=True)
)

if not all_audio:
    raise FileNotFoundError(
        f"Nenhum arquivo de áudio encontrado em: {DRIVE_INPUT_DIR}\n"
        f"Formatos suportados: {', '.join(AUDIO_EXTENSIONS)}"
    )

# --- lê as anotações ----------------------------------------------------------
annotations_by_key = {}
missing_annotations = []
ambiguous_annotations = []

if ANNOTATION_FORMAT == 'csv':
    if not os.path.exists(DRIVE_ANNOTATION_CSV):
        raise FileNotFoundError(
            f"Tabela de anotações não encontrada: {DRIVE_ANNOTATION_CSV}\n"
            "Confira o caminho no formulário de Anotações (Etapa 2)."
        )
    csv_annotations, csv_aliases = read_annotation_csv(DRIVE_ANNOTATION_CSV)
    for audio_path in all_audio:
        key = recording_key(audio_path, DRIVE_INPUT_DIR)
        name = key.rsplit('/', 1)[-1]
        if key in csv_annotations:
            annotations_by_key[key] = csv_annotations[key]
        elif csv_aliases.get(name):
            # A tabela identificou a gravação apenas pelo nome do arquivo, e só
            # uma gravação nela atende por esse nome.
            annotations_by_key[key] = csv_annotations[csv_aliases[name]]
        elif name in csv_aliases:
            ambiguous_annotations.append((key, f"'{name}' identifica várias gravações na tabela"))
else:
    if not os.path.isdir(DRIVE_ANNOTATION_DIR):
        raise FileNotFoundError(
            f"Pasta de anotações não encontrada: {DRIVE_ANNOTATION_DIR}\n"
            "Confira o caminho no formulário de Anotações (Etapa 2)."
        )
    annotation_files = sorted(
        path for path in _glob.glob(os.path.join(DRIVE_ANNOTATION_DIR, '**', '*'), recursive=True)
        if os.path.isfile(path) and not path.lower().endswith(AUDIO_EXTENSIONS)
    )
    reader = read_raven_table if ANNOTATION_FORMAT == 'raven' else read_audacity_labels
    for audio_path in all_audio:
        stem = os.path.splitext(os.path.basename(audio_path))[0]
        relative_dir = os.path.dirname(os.path.relpath(audio_path, DRIVE_INPUT_DIR))
        matches = [path for path in annotation_files
                   if name_matches(os.path.basename(path), stem)]
        # Prefere arquivos de anotação que estejam na mesma subpasta da gravação,
        # para que um nome repetido entre subpastas seja resolvido pelo seu próprio.
        same_folder = [path for path in matches
                       if os.path.dirname(os.path.relpath(path, DRIVE_ANNOTATION_DIR))
                       == relative_dir]
        chosen = same_folder or matches
        if not chosen:
            continue
        key = recording_key(audio_path, DRIVE_INPUT_DIR)
        if not same_folder and len({os.path.dirname(path) for path in chosen}) > 1:
            # Mesmo nome de arquivo em várias pastas de anotação e nenhuma delas
            # espelha a pasta do áudio: juntá-las inventaria anotações.
            ambiguous_annotations.append(
                (key, 'corresponde a ' + ', '.join(os.path.relpath(p, DRIVE_ANNOTATION_DIR)
                                                   for p in chosen)))
            continue
        found = []
        for path in chosen:
            found.extend(reader(path))
        annotations_by_key[key] = found

# --- pareia áudio com anotações -----------------------------------------------
validation_files = []
for audio_path in all_audio:
    key = recording_key(audio_path, DRIVE_INPUT_DIR)
    if key in annotations_by_key:
        found = annotations_by_key[key]
    elif INCLUDE_UNANNOTATED_FILES and not any(k == key for k, _ in ambiguous_annotations):
        found = []
    else:
        if not any(k == key for k, _ in ambiguous_annotations):
            missing_annotations.append(key)
        continue
    validation_files.append({'path': audio_path, 'key': key, 'annotations': found})

if not validation_files:
    raise FileNotFoundError(
        "Nenhuma gravação pôde ser pareada com anotações.\n"
        f"Arquivos de áudio encontrados: {len(all_audio)}  |  Conjuntos de anotações lidos: {len(annotations_by_key)}\n"
        + (
            f"{len(ambiguous_annotations)} gravação(ões) corresponderam a vários conjuntos de "
            "anotações e nenhuma pôde ser resolvida — ex.: "
            f"'{ambiguous_annotations[0][0]}' {ambiguous_annotations[0][1]}.\n"
            "Espelhe as subpastas da pasta de áudio na pasta de anotações, ou identifique as "
            "gravações pelo caminho relativo completo."
            if ambiguous_annotations else
            "Confira se os nomes dos arquivos de anotação começam com o nome do arquivo de áudio "
            "(sem a extensão), ou ative INCLUDE_UNANNOTATED_FILES se as gravações realmente não "
            "têm nenhuma ocorrência."
        )
    )

# --- os rótulos a validar -----------------------------------------------------
annotation_label_counts = defaultdict(int)
for entry in validation_files:
    for annotation in entry['annotations']:
        annotation_label_counts[annotation['label']] += 1

AVAILABLE_LABELS = sorted(annotation_label_counts)
if not AVAILABLE_LABELS and not SELECTED_LABELS:
    raise ValueError(
        "Nenhum rótulo a validar: as anotações ficaram vazias após a filtragem.\n"
        "Confira IGNORE_ANNOTATION_LABELS e o nome da coluna de rótulo no formulário de Anotações."
    )

# Seleção em branco significa "tudo o que o conjunto de dados tem"; uma seleção
# explícita prevalece, e pode nomear um rótulo que o conjunto nunca usou (avaliado
# apenas por falsos positivos).
EVAL_LABELS = sorted(set(SELECTED_LABELS)) if SELECTED_LABELS else AVAILABLE_LABELS

total_annotations = sum(annotation_label_counts.values())
print(f"Pasta de áudio             : {DRIVE_INPUT_DIR}")
print(f"Arquivos de áudio          : {len(all_audio)}")
print(f"Gravações a validar        : {len(validation_files)}")
print(f"Anotações lidas            : {total_annotations}")
print()

print(f"Rótulos encontrados nas anotações ({len(AVAILABLE_LABELS)}):")
for label in AVAILABLE_LABELS:
    note = '' if label in EVAL_LABELS else '   ← não selecionado, ignorado'
    print(f"  {label:<45} {annotation_label_counts[label]:>6} anotação(ões){note}")

print()
if SELECTED_LABELS:
    skipped = [label for label in AVAILABLE_LABELS if label not in EVAL_LABELS]
    print(f"Validando {len(EVAL_LABELS)} rótulo(s) selecionado(s):")
    for label in EVAL_LABELS:
        count = annotation_label_counts.get(label, 0)
        note  = f'{count:>6} anotação(ões)' if count else '     apenas falsos positivos (sem anotações)'
        print(f"  {label:<45} {note}")
    if skipped:
        print(f"  ({len(skipped)} rótulo(s) anotado(s) fora desta execução: {', '.join(skipped[:6])}"
              f"{' ...' if len(skipped) > 6 else ''})")
    # Um rótulo selecionado sem anotações que se parece muito com um que tem
    # anotações é quase sempre erro de digitação ou tradução não configurada.
    for label in EVAL_LABELS:
        if annotation_label_counts.get(label):
            continue
        close = close_names(label, AVAILABLE_LABELS, cutoff=0.75)
        if close:
            print(f"  NOTA: '{label}' não correspondeu a nenhuma anotação, mas se parece com {close}.")
            print(f"        Corrija a grafia em VALIDATION_LABELS, ou adicione uma tradução")
            print(f"        como '{close[0]}={label}' em TRANSLATE_ANNOTATION_LABELS.")
else:
    print("Validando todos os rótulos acima.")
    print("Para validar apenas alguns, liste-os em VALIDATION_LABELS (formulário de Anotações, Etapa 2).")

unused_translations = sorted(set(ANNOTATION_LABEL_MAP_D) - set(raw_label_counts))
if unused_translations:
    print()
    print("ATENÇÃO: estas traduções de anotação nunca corresponderam a nenhum rótulo:")
    for source in unused_translations:
        close = close_names(source, sorted(raw_label_counts), n=2)
        print(f"  '{source}'" + (f"   — você quis dizer {close}?" if close else ''))

if missing_annotations:
    print()
    print(f"ATENÇÃO: {len(missing_annotations)} gravação(ões) não tinham arquivo de anotação e foram ignoradas:")
    for name in missing_annotations[:5]:
        print(f"  {name}")
    if len(missing_annotations) > 5:
        print(f"  ... e mais {len(missing_annotations) - 5}")

if ambiguous_annotations:
    print()
    print(f"ATENÇÃO: {len(ambiguous_annotations)} gravação(ões) não puderam ser associadas a um "
          f"único conjunto de anotações e foram ignoradas:")
    for key, reason in ambiguous_annotations[:5]:
        print(f"  {key}  —  {reason}")
    if len(ambiguous_annotations) > 5:
        print(f"  ... e mais {len(ambiguous_annotations) - 5}")
    print("  Espelhe as subpastas da pasta de áudio na pasta de anotações, ou identifique as")
    print("  gravações pelo caminho relativo completo, para que cada uma resolva o seu conjunto.")

print()
print("Variação de parâmetros concluída. Continue para a Etapa 4.")

---
## Etapa 4 — Carregar o modelo e seus rótulos

Esta célula carrega seu modelo e lê a lista de classes que ele consegue detectar, e então confere
se os rótulos que você está validando realmente existem no modelo.

**Formato do arquivo de rótulos:** um arquivo de texto simples com um rótulo por linha, por exemplo:
```
Phyllodytes luteolus
Dendropsophus minutus
Ruído de fundo
```

O número de linhas do arquivo de rótulos precisa ser igual ao número de saídas do seu modelo.

In [ ]:
#@title 🧠 Carregar modelo e rótulos { display-mode: "form" }

import numpy as np

_DELIMITERS = {"tab": "\t", "comma (,)": ",", "semicolon (;)": ";", "underscore (_)": "_"}
_labels_sep = _DELIMITERS.get(LABELS_DELIMITER, "\t")

if MODEL_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    print(f'Baixando o modelo do HuggingFace: {HF_REPO_ID} / {HF_MODEL_FILE} ...')
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_MODEL_FILE)
    _labels_repo = HF_LABELS_REPO.strip() or HF_REPO_ID
    print(f'Baixando os rótulos do HuggingFace: {_labels_repo} / {HF_LABELS_FILE} ...')
    labels_path = hf_hub_download(repo_id=_labels_repo, filename=HF_LABELS_FILE)
elif MODEL_SOURCE == 'google_drive':
    model_path  = DRIVE_MODEL_PATH
    labels_path = DRIVE_LABELS_PATH
else:
    raise ValueError(f"MODEL_SOURCE precisa ser 'huggingface' ou 'google_drive', recebido: {MODEL_SOURCE!r}")

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Arquivo do modelo não encontrado: {model_path}\n"
        "Confira o caminho no formulário de Modelo (Etapa 2)."
    )
if not os.path.exists(labels_path):
    raise FileNotFoundError(
        f"Arquivo de rótulos não encontrado: {labels_path}\n"
        "Confira o caminho no formulário de Modelo (Etapa 2)."
    )

ext = os.path.splitext(model_path)[1].lower()
if ext == '.tflite':
    MODEL_TYPE = 'tflite'
elif ext == '.onnx':
    MODEL_TYPE = 'onnx'
else:
    raise ValueError(f"Formato de modelo não suportado: '{ext}'.\nO arquivo do modelo precisa terminar em .tflite ou .onnx")

MODEL_NAME = os.path.splitext(os.path.basename(model_path))[0]

if MODEL_TYPE == 'tflite':
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    model = TFLiteInterpreter(model_path=model_path)
    model.allocate_tensors()
    in_shape  = model.get_input_details()[0]['shape']
    out_shape = model.get_output_details()[0]['shape']
    print('Modelo TFLite carregado.')
    print(f'  Formato de entrada : {in_shape}')
    print(f'  Formato de saída   : {out_shape}')
    if globals().get('COMPUTE_DEVICE', 'CPU') == 'GPU':
        print('  Nota               : modelos TFLite rodam apenas na CPU; a configuração de GPU é ignorada.')
elif MODEL_TYPE == 'onnx':
    import onnxruntime as ort

    # Seleciona o provedor de execução conforme o dispositivo escolhido na Etapa 1.
    _device    = globals().get('COMPUTE_DEVICE', 'CPU')
    _available = ort.get_available_providers()
    if _device == 'GPU' and 'CUDAExecutionProvider' in _available:
        _providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    else:
        if _device == 'GPU':
            print('ATENÇÃO: GPU solicitada, mas o CUDAExecutionProvider não está disponível — voltando para CPU.')
            print('         Troque o ambiente do Colab para GPU (Ambiente de execução → Alterar o tipo → T4 GPU)')
            print('         e reexecute a partir da Etapa 1 para usar a GPU.')
        _providers = ['CPUExecutionProvider']

    model = ort.InferenceSession(model_path, providers=_providers)
    in_shape  = model.get_inputs()[0].shape
    out_shape = model.get_outputs()[0].shape
    print('Modelo ONNX carregado.')
    print(f'  Formato de entrada : {in_shape}')
    print(f'  Formato de saída   : {out_shape}')
    print(f'  Dispositivo        : {_device}  ({model.get_providers()})')

labels = []
with open(labels_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

if LABELS_HAS_HEADER and lines:
    lines = lines[1:]

for line in lines:
    line = line.strip()
    if not line:
        continue
    if _labels_sep in line:
        parts = line.split(_labels_sep)
        if LABELS_COLUMN_INDEX < len(parts):
            labels.append(parts[LABELS_COLUMN_INDEX].strip())
        else:
            print(f"  ATENÇÃO: índice de coluna {LABELS_COLUMN_INDEX} não encontrado na linha: {line!r} — ignorando.")
    else:
        labels.append(line)

print(f'\nRótulos carregados: {len(labels)} classes')
print(f'  Delimitador : {LABELS_DELIMITER}  |  Índice da coluna : {LABELS_COLUMN_INDEX}  |  Cabeçalho ignorado : {LABELS_HAS_HEADER}')
print(f'  Primeiros 5 : {labels[:5]}')
print(f'  Nome do modelo: {MODEL_NAME}')

# --- associa as saídas do modelo aos rótulos sendo avaliados ------------------
# Apenas estas colunas da saída do modelo são mantidas, o que é o que torna barato
# guardar em cache os logits de cada gravação mesmo para um modelo de 6000 classes.
# Vários rótulos do modelo podem cair sobre um mesmo rótulo avaliado; vence o de
# maior pontuação.
EVAL_COLUMNS = {label: [] for label in EVAL_LABELS}
for index, model_label in enumerate(labels):
    canonical = MODEL_LABEL_MAP_D.get(model_label, model_label)
    if canonical in EVAL_COLUMNS:
        EVAL_COLUMNS[canonical].append(index)

unpredictable = [label for label, columns in EVAL_COLUMNS.items() if not columns]
print()
print(f'Rótulos a validar associados a saídas do modelo: '
      f'{len(EVAL_LABELS) - len(unpredictable)}/{len(EVAL_LABELS)}')
for label in EVAL_LABELS:
    if EVAL_COLUMNS[label]:
        aliases = [labels[i] for i in EVAL_COLUMNS[label]]
        detail = f'  ← {aliases}' if aliases != [label] else ''
        print(f'  {label:<45} ok{detail}')

if unpredictable:
    # O que o modelo consegue de fato produzir, após a tradução do próprio modelo.
    model_vocabulary = sorted({MODEL_LABEL_MAP_D.get(name, name) for name in labels})
    print()
    print('ATENÇÃO: estes rótulos não têm saída correspondente no modelo. O modelo nunca')
    print('         consegue prevê-los, então a precisão fica indefinida e o recall fica em 0.')
    for label in unpredictable:
        print(f'  {label}')
        close = close_names(label, model_vocabulary)
        if close:
            print(f'      rótulo(s) mais próximo(s) no modelo : {close}')
            # Traduzir o lado do modelo mantém a sua grafia nos resultados; traduzir
            # o lado da anotação renomearia o seu rótulo para o do modelo, o que
            # raramente é o desejado em um relatório.
            print( '      para ligá-los, adicione isto em TRANSLATE_MODEL_LABELS (Etapa 2):')
            print(f'        {close[0]}={label}')
        else:
            print('      nenhum rótulo parecido no modelo — confira se este modelo realmente '
                  'cobre este rótulo.')

---
## Etapa 5 — Executar o modelo uma vez e armazenar os logits

Esta é a única etapa cara. Para cada gravação:
1. O áudio é copiado do Drive, decodificado e pré-processado
2. Ele é dividido em janelas (ex.: 3 segundos cada)
3. Cada janela é enviada ao modelo e suas **saídas brutas (logits)** são guardadas — sem limiar,
   sem ativação, sem descartar nada

Como os logits ficam armazenados, a variação de parâmetros da Etapa 6 custa milissegundos por ponto de operação
em vez de uma reexecução completa do modelo. Se o cache de logits estiver ativado na Etapa 2, eles
também são gravados no seu Drive, de modo que uma sessão futura possa pular esta etapa por completo.

> Dependendo da quantidade e da duração das suas gravações, esta etapa pode demorar. O progresso é
> mostrado abaixo.

In [ ]:
#@title 🚀 Extrair logits { display-mode: "form" }

#@markdown **Tamanho do lote** — janelas enviadas ao modelo de uma vez. Lotes maiores ajudam muito
#@markdown em ambientes com GPU e modelos ONNX. Modelos sem eixo de lote dinâmico voltam
#@markdown automaticamente a processar uma janela por vez.
BATCH_SIZE = 32  #@param {type:"integer"}
BATCH_SIZE = max(1, int(BATCH_SIZE))

import hashlib
import shutil
import time
import librosa

segment_samples = int(SEGMENT_DURATION_S * SAMPLE_RATE)
stride_samples  = max(1, int(segment_samples * (1 - SEGMENT_OVERLAP)))

# Qualquer mudança nestes valores invalida um arquivo de logits em cache — os
# números dentro dele deixariam de descrever a execução que está sendo pedida.
CACHE_SIGNATURE = hashlib.sha1(repr((
    os.path.abspath(model_path), MODEL_NAME, SAMPLE_RATE, SEGMENT_DURATION_S, SEGMENT_OVERLAP,
    FILTER_TYPE, FILTER_LOW_HZ, FILTER_HIGH_HZ, AUDIO_SPEED, tuple(EVAL_LABELS),
    tuple(tuple(EVAL_COLUMNS[label]) for label in EVAL_LABELS),
)).encode()).hexdigest()[:16]


def preprocess_audio(audio, sr):
    if FILTER_TYPE != 'none':
        from scipy.signal import butter, sosfilt
        nyq = sr / 2.0
        if FILTER_TYPE == 'lowpass':
            sos = butter(5, min(FILTER_HIGH_HZ, nyq - 1) / nyq, btype='low', output='sos')
        elif FILTER_TYPE == 'highpass':
            sos = butter(5, max(FILTER_LOW_HZ, 1) / nyq, btype='high', output='sos')
        elif FILTER_TYPE == 'bandpass':
            lo = max(FILTER_LOW_HZ, 1) / nyq
            hi = min(FILTER_HIGH_HZ, nyq - 1) / nyq
            sos = butter(5, [lo, hi], btype='band', output='sos')
        audio = sosfilt(sos, audio).astype(np.float32)
    if AUDIO_SPEED != 1.0:
        audio = librosa.effects.time_stretch(audio, rate=AUDIO_SPEED)
    return audio


_batch_supported = [None]  # testado sob demanda no primeiro lote real


def run_model_batch(segments):
    """Executa um array (n_janelas, n_amostras) float32 através do modelo.

    Devolve os logits brutos como (n_janelas, n_saídas). Volta a processar uma
    janela por vez em modelos sem eixo de lote, de modo que um grafo TFLite de
    lote fixo continue funcionando — apenas mais devagar.
    """
    segments = np.ascontiguousarray(segments, dtype=np.float32)

    if MODEL_TYPE == 'onnx' and _batch_supported[0] is not False and len(segments) > 1:
        input_name = model.get_inputs()[0].name
        try:
            out = model.run(None, {input_name: segments})[0]
            _batch_supported[0] = True
            return np.asarray(out, dtype=np.float32).reshape(len(segments), -1)
        except Exception:
            if _batch_supported[0] is None:
                print('  Nota: este modelo não tem eixo de lote dinâmico — processando uma janela por vez.')
            _batch_supported[0] = False

    outputs = []
    for segment in segments:
        if MODEL_TYPE == 'tflite':
            in_det  = model.get_input_details()[0]
            out_det = model.get_output_details()[0]
            try:
                model.set_tensor(in_det['index'], segment.reshape(in_det['shape']))
            except ValueError:
                model.set_tensor(in_det['index'], segment.reshape(1, -1))
            model.invoke()
            outputs.append(model.get_tensor(out_det['index']).flatten())
        else:
            input_name = model.get_inputs()[0].name
            try:
                out = model.run(None, {input_name: segment.reshape(1, -1)})[0]
            except Exception:
                out = model.run(None, {input_name: segment[np.newaxis, :]})[0]
            outputs.append(np.asarray(out, dtype=np.float32).flatten())
    return np.stack(outputs).astype(np.float32)


def select_eval_columns(logits):
    """Reduz os logits completos do modelo a uma coluna por rótulo avaliado.

    Quando vários rótulos do modelo caem sobre o mesmo rótulo avaliado, vence o
    maior logit. Com sensibilidade do sigmoide negativa a ativação é monótona
    crescente, então o maior logit também é a maior pontuação em qualquer limiar
    e bias da variação de parâmetros — a redução pode ser feita aqui com segurança, antes de
    qualquer ativação.
    """
    selected = np.full((len(logits), len(EVAL_LABELS)), -1e9, dtype=np.float32)
    for column, label in enumerate(EVAL_LABELS):
        indices = [i for i in EVAL_COLUMNS[label] if i < logits.shape[1]]
        if indices:
            selected[:, column] = logits[:, indices].max(axis=1)
    return selected


def cache_path_for(entry):
    # A chave carrega a subpasta, então duas gravações que compartilham o nome do
    # arquivo entre subpastas ainda ganham entradas de cache próprias.
    safe = ''.join(c if c.isalnum() or c in '._-' else '_' for c in entry['key'])
    return os.path.join(DRIVE_LOGITS_CACHE_DIR, f'{safe}.{MODEL_NAME}.logits.npz')


def load_cached_logits(entry):
    """Devolve (starts, ends, logits) do cache, ou None se não for utilizável."""
    path = cache_path_for(entry)
    if not os.path.exists(path):
        return None
    try:
        with np.load(path, allow_pickle=False) as data:
            if str(data['signature']) != CACHE_SIGNATURE:
                return None
            return data['starts'], data['ends'], data['logits']
    except Exception as error:
        print(f'  ATENÇÃO: não foi possível ler o cache {os.path.basename(path)} ({error}) — recalculando.')
        return None


LOCAL_AUDIO_TMP = '/content/audio_tmp'
_last_remount = [0.0]


def copy_from_drive(src, dst, retries=3, remount_cooldown=30):
    """Copia com novas tentativas; uma montagem FUSE morta precisa ser remontada."""
    last_error = None
    for attempt in range(retries):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as error:
            last_error = error
            # Um erro FUSE 'Transport endpoint is not connected' significa que a
            # própria montagem caiu e só tentar de novo nunca vai funcionar. O
            # intervalo mínimo evita que uma sequência de arquivos ilegíveis
            # dispare remontagens sem parar.
            if time.time() - _last_remount[0] > remount_cooldown:
                print(f'  ATENÇÃO: falha ao ler do Google Drive ({error}) — remontando e tentando de novo...')
                try:
                    from google.colab import drive
                    drive.mount('/content/drive', force_remount=True)
                except Exception as remount_error:
                    print(f'  ATENÇÃO: a remontagem falhou: {remount_error}')
                _last_remount[0] = time.time()
            time.sleep(2 * (attempt + 1))
    raise last_error


def extract_logits(entry):
    """Decodifica uma gravação, divide em janelas e devolve (starts, ends, logits)."""
    os.makedirs(LOCAL_AUDIO_TMP, exist_ok=True)
    local_path = os.path.join(LOCAL_AUDIO_TMP, os.path.basename(entry['path']))
    try:
        copy_from_drive(entry['path'], local_path)
        audio, _ = librosa.load(local_path, sr=SAMPLE_RATE, mono=True)
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)

    audio = preprocess_audio(audio, SAMPLE_RATE)

    starts, segments = [], []
    for start_sample in range(0, len(audio), stride_samples):
        segment = audio[start_sample:start_sample + segment_samples]
        if len(segment) < segment_samples * 0.5:
            continue
        if len(segment) < segment_samples:
            segment = np.pad(segment, (0, segment_samples - len(segment)))
        starts.append(start_sample / SAMPLE_RATE)
        segments.append(segment.astype(np.float32))

    if not segments:
        return np.zeros(0, np.float32), np.zeros(0, np.float32), \
               np.zeros((0, len(EVAL_LABELS)), np.float32)

    chunks = [run_model_batch(np.stack(segments[i:i + BATCH_SIZE]))
              for i in range(0, len(segments), BATCH_SIZE)]
    logits = select_eval_columns(np.concatenate(chunks, axis=0))

    # O pré-processamento pode ter esticado o áudio; as anotações são sempre dadas
    # na linha do tempo da gravação original, então convertemos de volta os limites
    # das janelas.
    starts = np.asarray(starts, dtype=np.float32) * AUDIO_SPEED
    ends   = starts + np.float32(SEGMENT_DURATION_S * AUDIO_SPEED)
    return starts, ends, logits


print(f'Extraindo logits de {len(validation_files)} gravação(ões) com o modelo "{MODEL_NAME}"')
print(f'Rótulos avaliados : {len(EVAL_LABELS)}  |  Tamanho do lote: {BATCH_SIZE}')
print(f'Janela            : {SEGMENT_DURATION_S}s  |  Sobreposição: {SEGMENT_OVERLAP}  |  {SAMPLE_RATE} Hz')
print(f'Cache de logits   : {DRIVE_LOGITS_CACHE_DIR if USE_LOGITS_CACHE else "desativado"}')
print('=' * 70)

n_cached = n_computed = n_failed = 0
total_windows = 0
run_start = time.time()

for index, entry in enumerate(validation_files, 1):
    name = os.path.basename(entry['path'])
    cached = load_cached_logits(entry) if USE_LOGITS_CACHE else None

    if cached is not None:
        entry['starts'], entry['ends'], entry['logits'] = cached
        n_cached += 1
        print(f"[{index}/{len(validation_files)}] {name}  →  {len(entry['starts'])} janelas (do cache)")
    else:
        file_start = time.time()
        try:
            entry['starts'], entry['ends'], entry['logits'] = extract_logits(entry)
        except Exception as error:
            print(f'[{index}/{len(validation_files)}] {name}  ERRO: {error} — ignorando.')
            entry['logits'] = None
            n_failed += 1
            continue
        n_computed += 1
        elapsed = time.time() - file_start
        print(f"[{index}/{len(validation_files)}] {name}  →  {len(entry['starts'])} janelas  "
              f"({elapsed:.1f}s)")
        if USE_LOGITS_CACHE:
            try:
                np.savez_compressed(cache_path_for(entry), starts=entry['starts'],
                                    ends=entry['ends'], logits=entry['logits'],
                                    signature=np.array(CACHE_SIGNATURE))
            except Exception as error:
                print(f'  ATENÇÃO: não foi possível gravar o cache: {error}')

    total_windows += len(entry['starts'])

validation_files = [entry for entry in validation_files if entry.get('logits') is not None]
if not validation_files:
    raise RuntimeError('Nenhuma gravação pôde ser analisada — veja os erros acima.')

print()
print('=' * 70)
print('Extração de logits concluída.')
print(f'  Gravações  : {len(validation_files)}  ({n_computed} analisadas, {n_cached} do cache, {n_failed} com falha)')
print(f'  Janelas    : {total_windows}')
print(f'  Tempo total: {time.time() - run_start:.1f}s')
print(f'  Na memória : {sum(e["logits"].nbytes for e in validation_files) / 1e6:.1f} MB de logits')

---
## Etapa 6 — Varrer os pontos de operação e pontuá-los

Agora a parte barata. Para cada combinação de **limiar de pontuação** × **bias do sigmoide**:

1. Os logits armazenados são ativados em pontuações: `pontuação = 1 / (1 + exp(sensibilidade × (logit + 10 × (bias − 1))))`
2. Toda pontuação maior ou igual ao limiar vira uma detecção
3. As detecções são comparadas com as anotações para produzir contagens de **TP / FP / FN**
4. As contagens viram **precisão**, **recall** e **F1**, por rótulo, mais linhas de resumo com as
   médias micro e macro

**Como a contagem funciona**, conforme a granularidade escolhida na Etapa 2:

| Granularidade | TP | FP | FN |
|---|---|---|---|
| `annotation` | uma anotação com ao menos uma detecção sobreposta do mesmo rótulo (cada detecção é reivindicada por no máximo uma anotação) | uma detecção que não cai dentro de nenhuma anotação do mesmo rótulo | uma anotação sem detecção sobreposta ainda não reivindicada |
| `window` | uma janela em que o rótulo foi detectado e anotado | uma janela em que o rótulo foi detectado mas não anotado | cada anotação sobreposta a uma janela em que o rótulo não foi detectado |
| `file` | uma gravação em que o rótulo foi detectado e anotado | detectado mas nunca anotado naquela gravação | anotado mas nunca detectado naquela gravação |

Uma métrica indefinida — precisão para um rótulo que o modelo nunca previu, recall para um rótulo
que nunca foi anotado — fica **vazia** em vez de virar 0, mas conta como 0 na média macro (a mesma
convenção usada pelo scikit-learn).

In [ ]:
#@title 🎚️ Variação de parâmetros { display-mode: "form" }

MICRO_AVERAGE_LABEL = '__micro_avg__'
MACRO_AVERAGE_LABEL = '__macro_avg__'


def windows_match(window_starts, window_ends, ann_start, ann_end):
    """Máscara booleana sobre as janelas: quais correspondem a esta anotação?"""
    overlap = np.minimum(window_ends, ann_end) - np.maximum(window_starts, ann_start)
    if MATCH_MODE == 'overlap':
        return overlap > 0
    overlap = np.maximum(overlap, 0.0)
    union   = (window_ends - window_starts) + (ann_end - ann_start) - overlap
    with np.errstate(divide='ignore', invalid='ignore'):
        iou = np.where(union > 0, overlap / union, 0.0)
    return iou >= IOU_THRESHOLD


def build_match_index(entry):
    """Pré-calcula tudo sobre a correspondência anotação ↔ janela, uma vez por arquivo.

    Nada disso depende do limiar ou do bias, então é calculado uma única vez e
    reaproveitado em todos os pontos de operação da variação de parâmetros.
    """
    starts, ends = entry['starts'], entry['ends']
    n_windows    = len(starts)

    # n_ann[w, l] — quantas anotações do rótulo l correspondem à janela w.
    n_ann = np.zeros((n_windows, len(EVAL_LABELS)), dtype=np.int32)
    # por rótulo: para cada anotação, as janelas que ela alcança (na ordem das anotações)
    ann_windows = {label: [] for label in EVAL_LABELS}
    # por rótulo: quantas anotações dele existem nesta gravação ao todo
    ann_totals = {label: 0 for label in EVAL_LABELS}

    label_column = {label: index for index, label in enumerate(EVAL_LABELS)}
    ordered = sorted(entry['annotations'], key=lambda a: (a['start_time'], a['end_time']))
    for annotation in ordered:
        column = label_column.get(annotation['label'])
        if column is None:  # não é um rótulo avaliado
            continue
        ann_totals[annotation['label']] += 1
        matched = (windows_match(starts, ends, annotation['start_time'], annotation['end_time'])
                   if n_windows else np.zeros(0, dtype=bool))
        n_ann[matched, column] += 1
        ann_windows[annotation['label']].append(np.flatnonzero(matched))

    entry['n_ann']       = n_ann
    entry['has_ann']     = n_ann > 0
    entry['ann_windows'] = ann_windows
    entry['ann_totals']  = ann_totals


def activate(logits, bias):
    """Ativação sigmoide com a parametrização de bias do auricularia."""
    transformed_bias = (bias - 1.0) * 10.0
    return 1.0 / (1.0 + np.exp(
        SIGMOID_SENSITIVITY * np.clip(logits + transformed_bias, -20, 20)
    ))


def count_window_level(entry, detected):
    """TP/FP/FN por rótulo, uma contagem por janela do modelo × rótulo."""
    has_ann = entry['has_ann']
    tp = np.count_nonzero(detected & has_ann, axis=0)
    fp = np.count_nonzero(detected & ~has_ann, axis=0)
    # Cada anotação sobreposta a uma janela sem detecção é um FN, então um canto
    # longo perdido pelo modelo custa tantos FNs quantas janelas ele abrange.
    fn = np.where(detected, 0, entry['n_ann']).sum(axis=0)
    # Uma anotação que nenhuma janela alcançou (ex.: está além do fim do áudio
    # decodificado) sumiria das contagens por completo.
    for column, label in enumerate(EVAL_LABELS):
        if entry['n_ann'][:, column].sum() == 0:
            fn[column] += entry['ann_totals'][label]
    return tp, fp, fn, np.zeros(len(EVAL_LABELS), dtype=np.int64)


def count_annotation_level(entry, detected, scores):
    """TP/FP/FN por rótulo, uma contagem por evento anotado."""
    n_labels = len(EVAL_LABELS)
    tp = np.zeros(n_labels, dtype=np.int64)
    fn = np.zeros(n_labels, dtype=np.int64)

    for column, label in enumerate(EVAL_LABELS):
        claimed = set()
        for window_indices in entry['ann_windows'][label]:
            # Anotação mais antiga primeiro (build_match_index já ordenou), e cada
            # detecção só pode ser reivindicada uma vez, então uma sequência de
            # detecções dentro de um mesmo canto anotado vira um único TP.
            unclaimed = [w for w in window_indices
                         if detected[w, column] and w not in claimed]
            if not unclaimed:
                fn[column] += 1
                continue
            best = max(unclaimed, key=lambda w: scores[w, column])
            claimed.add(best)
            tp[column] += 1

    # Uma detecção sobreposta a uma anotação de outro rótulo é uma confusão real e
    # continua contando como falso positivo.
    fp = np.count_nonzero(detected & ~entry['has_ann'], axis=0)
    return tp, fp, fn, np.zeros(n_labels, dtype=np.int64)


def count_file_level(entry, detected):
    """TP/FP/FN/TN por rótulo, uma contagem por gravação: esteve presente ou não?"""
    in_model = detected.any(axis=0)
    in_annotations = np.array([entry['ann_totals'][label] > 0 for label in EVAL_LABELS])
    tp = (in_annotations & in_model).astype(np.int64)
    fn = (in_annotations & ~in_model).astype(np.int64)
    fp = (~in_annotations & in_model).astype(np.int64)
    tn = (~in_annotations & ~in_model).astype(np.int64) if REPORT_TN \
        else np.zeros(len(EVAL_LABELS), dtype=np.int64)
    return tp, fp, fn, tn


def safe_divide(numerator, denominator):
    """Devolve None em vez de erro, para que uma métrica indefinida fique como célula vazia."""
    return None if denominator == 0 else numerator / denominator


def metrics_from_counts(tp, fp, fn):
    precision = safe_divide(tp, tp + fp)
    recall    = safe_divide(tp, tp + fn)
    if precision is None or recall is None or (precision + recall) == 0:
        f1 = None
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


print('Indexando as anotações contra as janelas do modelo...')
for entry in validation_files:
    build_match_index(entry)

print(f'Varrendo {len(THRESHOLDS)} limiar(es) × {len(BIASES)} bias '
      f'= {len(THRESHOLDS) * len(BIASES)} ponto(s) de operação')
print('=' * 70)

metric_rows = []
sweep_start = time.time()

for bias in BIASES:
    # O modelo rodou uma vez; cada bias é uma passada de aritmética sobre os logits guardados.
    activated = [activate(entry['logits'], bias) for entry in validation_files]

    for threshold in THRESHOLDS:
        totals = {key: np.zeros(len(EVAL_LABELS), dtype=np.int64)
                  for key in ('TP', 'FP', 'FN', 'TN')}

        for entry, scores in zip(validation_files, activated):
            detected = scores >= threshold
            if GRANULARITY == 'annotation':
                tp, fp, fn, tn = count_annotation_level(entry, detected, scores)
            elif GRANULARITY == 'file':
                tp, fp, fn, tn = count_file_level(entry, detected)
            else:
                tp, fp, fn, tn = count_window_level(entry, detected)
            totals['TP'] += tp
            totals['FP'] += fp
            totals['FN'] += fn
            totals['TN'] += tn

        variant = f"{MODEL_NAME}__st{threshold:g}_sb{bias:g}"
        macro = {'precision': [], 'recall': [], 'f1': []}

        for column, label in enumerate(EVAL_LABELS):
            tp, fp, fn, tn = (int(totals[key][column]) for key in ('TP', 'FP', 'FN', 'TN'))
            precision, recall, f1 = metrics_from_counts(tp, fp, fn)
            # Uma métrica indefinida conta como 0 na média macro: uma classe que o
            # modelo ignora deve puxar a média para baixo, não ser excluída dela.
            macro['precision'].append(precision or 0.0)
            macro['recall'].append(recall or 0.0)
            macro['f1'].append(f1 or 0.0)
            metric_rows.append({
                'variant': variant, 'score_threshold': threshold, 'sigmoid_bias': bias,
                'label': label, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                'support': tp + fn, 'precision': precision, 'recall': recall, 'f1': f1,
            })

        pooled = {key: int(totals[key].sum()) for key in ('TP', 'FP', 'FN', 'TN')}
        precision, recall, f1 = metrics_from_counts(pooled['TP'], pooled['FP'], pooled['FN'])
        metric_rows.append({
            'variant': variant, 'score_threshold': threshold, 'sigmoid_bias': bias,
            'label': MICRO_AVERAGE_LABEL, **pooled,
            'support': pooled['TP'] + pooled['FN'],
            'precision': precision, 'recall': recall, 'f1': f1,
        })
        metric_rows.append({
            'variant': variant, 'score_threshold': threshold, 'sigmoid_bias': bias,
            'label': MACRO_AVERAGE_LABEL, 'TP': None, 'FP': None, 'FN': None, 'TN': None,
            'support': len(EVAL_LABELS),
            **{key: sum(values) / len(values) for key, values in macro.items()},
        })

    print(f'  bias {bias:<5g} concluído  ({len(THRESHOLDS)} limiar(es))')

METRICS_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.validation_metrics.csv')
METRICS_COLUMNS = ['variant', 'score_threshold', 'sigmoid_bias', 'label',
                   'TP', 'FP', 'FN', 'TN', 'support', 'precision', 'recall', 'f1']
with open(METRICS_CSV_PATH, 'w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=METRICS_COLUMNS)
    writer.writeheader()
    for row in metric_rows:
        writer.writerow({key: ('' if row[key] is None else row[key]) for key in METRICS_COLUMNS})

print()
print('=' * 70)
print('Variação de parâmetros concluída.')
print(f'  Pontos de operação : {len(THRESHOLDS) * len(BIASES)}')
print(f'  Linhas de métricas : {len(metric_rows)}')
print(f'  Tempo              : {time.time() - sweep_start:.1f}s')
print(f'  Salvo em           : {METRICS_CSV_PATH}')

---
## Etapa 7 — Curvas de precisão e recall

**Este é o resultado do notebook.** Uma figura por rótulo. Em cada figura:

- O **eixo x** é o limiar de pontuação
- **Linhas contínuas com círculos preenchidos** são a **precisão** — das detecções que o modelo
  fez, quantas estavam certas
- **Linhas tracejadas com círculos vazados** são o **recall** — dos cantos que realmente estavam
  ali, quantos o modelo encontrou
- **Uma cor por bias do sigmoide**, identificada na legenda

Leia as duas juntas: o limiar em que as duas linhas da mesma cor se cruzam é aproximadamente onde
precisão e recall se equilibram (o pico do F1 costuma ficar perto dali). Se a precisão for baixa e
plana em toda a faixa, nenhum limiar vai salvar aquela classe — o problema é o modelo ou o rótulo,
não o ponto de operação.

As figuras são salvas como arquivos PNG ao lado do CSV de métricas no seu Drive.

In [ ]:
#@title 📈 Traçar precisão e recall { display-mode: "form" }

#@markdown **Rótulos a plotar** — separados por ponto e vírgula. Deixe em branco para plotar todos
#@markdown os rótulos avaliados.
LABELS_TO_PLOT = ""  #@param {type:"string"}

#@markdown **Plotar também as médias** — acrescenta figuras para a média micro (contagens somadas
#@markdown entre os rótulos, então classes comuns dominam) e para a média macro (média simples
#@markdown entre os rótulos, então classes raras pesam igual).
PLOT_AVERAGES = True  #@param {type:"boolean"}

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Matizes distintos, atribuídos em ordem fixa e nunca reciclados. Os valores de
# bias são ordenados, mas uma rampa de um só matiz do claro ao escuro deixava
# curvas vizinhas parecidas demais, então aqui a identidade vence a ordem — a
# legenda é que carrega a ordem.
#
# Esta ordenação foi escolhida com um validador de paleta, e só é segura até
# QUATRO curvas. As curvas se cruzam, então quaisquer duas cores podem acabar lado
# a lado; medido sobre todos os pares nesta superfície:
#   4 matizes  pior par ΔE 9.2 daltonismo / 16.3 visão normal  — seguro
#   5 matizes  magenta↔laranja 12.9 visão normal               — abaixo do piso de 15
#   6 matizes  verde↔laranja 3.2 sob protanopia                — quase idênticos
#   8 matizes  vermelho↔laranja 7.1 visão normal               — visivelmente parecidos
# Além de quatro não há solução apenas de cor: qualquer matiz adicional colide com
# algum já em uso. A célula avisa em vez de fingir o contrário.
_HUES = ['#2a78d6',   # azul
         '#eb6834',   # laranja
         '#1baf7a',   # água
         '#4a3aa7',   # violeta
         '#e87ba4',   # magenta
         '#008300',   # verde
         '#eda100',   # amarelo
         '#e34948']   # vermelho
# Alternativa para uma variação de parâmetros com mais bias do que matizes: uma rampa de um
# só matiz, do claro ao escuro. A partir daí as curvas deixam de ser
# individualmente identificáveis — a célula diz isso em vez de inventar um nono matiz.
_RAMP    = ['#86b6ef', '#5598e7', '#3987e5', '#256abf', '#184f95', '#0d366b']
_INK     = '#0b0b0b'
_MUTED   = '#898781'
_GRID    = '#e1e0d9'
_SURFACE = '#fcfcfb'
_AXIS    = '#52514e'


SAFE_BIAS_COLOURS = 4


def bias_colors(count):
    """Uma cor por curva de bias, tomada em ordem fixa da paleta acima."""
    if count <= len(_HUES):
        if count > SAFE_BIAS_COLOURS:
            print(f'NOTA: {count} bias do sigmoide significam {count} pares de curvas por figura.')
            print(f'      Apenas as {SAFE_BIAS_COLOURS} primeiras cores são garantidamente '
                  f'distinguíveis onde as curvas se cruzam')
            print('      (e da 6ª em diante são quase idênticas para daltonismo vermelho-verde).')
            print('      Prefira 4 bias ou menos por execução, ou leia os números exatos no')
            print('      CSV de métricas em vez de na figura.')
        return _HUES[:count]

    print(f'NOTA: {count} bias do sigmoide é mais do que as {len(_HUES)} cores distintas')
    print('      disponíveis, então as curvas voltam a usar um só matiz do claro ao escuro e')
    print('      ficam difíceis de distinguir. Considere varrer menos bias.')
    anchors = [tuple(int(step[i:i + 2], 16) for i in (1, 3, 5)) for step in _RAMP]
    colors = []
    for index in range(count):
        position = index / (count - 1) * (len(anchors) - 1)
        low      = min(int(position), len(anchors) - 2)
        fraction = position - low
        colors.append('#%02x%02x%02x' % tuple(
            round(anchors[low][c] + fraction * (anchors[low + 1][c] - anchors[low][c]))
            for c in range(3)
        ))
    return colors


BIAS_COLORS = dict(zip(BIASES, bias_colors(len(BIASES))))


def style_axis(ax, title, subtitle):
    ax.set_xlabel('limiar de pontuação', color=_AXIS, fontsize=10)
    ax.set_ylabel('precisão  |  recall', color=_AXIS, fontsize=10)
    ax.set_title(title, color=_INK, fontsize=13, loc='left', pad=30)
    ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=_MUTED, fontsize=9, va='bottom')
    ax.set_ylim(-0.02, 1.02)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.grid(axis='y', color=_GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(colors=_MUTED, labelsize=9)
    for side, spine in ax.spines.items():
        spine.set_visible(side == 'bottom')
        spine.set_color('#c3c2b7')


def plot_label(label, rows, output_path):
    """Desenha as curvas de precisão/recall de um rótulo, um par colorido por bias."""
    fig, ax = plt.subplots(figsize=(7.5, 5), dpi=120)
    fig.patch.set_facecolor(_SURFACE)
    ax.set_facecolor(_SURFACE)

    drawn = 0
    for bias in BIASES:
        points = sorted((r for r in rows if r['sigmoid_bias'] == bias),
                        key=lambda r: r['score_threshold'])
        # Uma métrica indefinida é um resultado real para um limiar que suprimiu a
        # classe, mas não é plotável — esses pontos saem da curva.
        precision_points = [(r['score_threshold'], r['precision']) for r in points
                            if r['precision'] is not None]
        recall_points    = [(r['score_threshold'], r['recall']) for r in points
                            if r['recall'] is not None]
        color = BIAS_COLORS[bias]
        if precision_points:
            ax.plot(*zip(*precision_points), color=color, linewidth=1.8,
                    marker='o', markersize=6.5, markeredgecolor=_SURFACE,
                    markeredgewidth=1.2, zorder=3)
            drawn += 1
        if recall_points:
            ax.plot(*zip(*recall_points), color=color, linewidth=1.8, linestyle='--',
                    marker='o', markersize=6.5, markerfacecolor=_SURFACE,
                    markeredgecolor=color, markeredgewidth=1.6, zorder=3)
            drawn += 1

    if not drawn:
        plt.close(fig)
        return None

    support = max((r['support'] or 0) for r in rows)
    style_axis(ax, label, f'{support} anotação(ões) · nível {GRANULARITY} · {MODEL_NAME}')

    handles = [Line2D([], [], color=BIAS_COLORS[b], linewidth=2.4, label=f'bias {b:g}')
               for b in BIASES]
    handles += [
        Line2D([], [], color=_MUTED, linewidth=1.8, marker='o', markersize=6.5,
               markeredgecolor=_SURFACE, label='precisão'),
        Line2D([], [], color=_MUTED, linewidth=1.8, linestyle='--', marker='o', markersize=6.5,
               markerfacecolor=_SURFACE, markeredgecolor=_MUTED, label='recall'),
    ]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 1),
              frameon=False, fontsize=9, labelcolor=_INK)

    fig.tight_layout()
    fig.savefig(output_path, facecolor=_SURFACE, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return output_path


def safe_filename(label):
    return ''.join(c if c.isalnum() or c in '._-' else '-' for c in label)


requested = parse_list(LABELS_TO_PLOT) or list(EVAL_LABELS)
unknown   = [label for label in requested if label not in EVAL_LABELS]
if unknown:
    raise ValueError(f"Rótulo(s) não avaliado(s): {unknown}\nDisponíveis: {EVAL_LABELS}")
if PLOT_AVERAGES:
    requested = requested + [MICRO_AVERAGE_LABEL, MACRO_AVERAGE_LABEL]

FIGURES_DIR = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}_figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

written = []
for label in requested:
    rows = [r for r in metric_rows if r['label'] == label]
    path = plot_label(label, rows,
                      os.path.join(FIGURES_DIR, f'{MODEL_NAME}_{safe_filename(label)}.png'))
    if path:
        written.append(path)
    else:
        print(f"'{label}': nenhum ponto plotável — precisão e recall ficam indefinidos em todos os "
              f"limiares (o modelo nunca previu este rótulo, ou ele nunca foi anotado).")

print()
print(f'{len(written)} figura(s) salva(s) em: {FIGURES_DIR}')

---
## Etapa 8 — Melhor ponto de operação por rótulo

Uma curva mostra o compromisso inteiro; esta tabela escolhe um ponto dela. Para cada rótulo, ela
informa o limiar e o bias com o **maior F1** — o melhor equilíbrio entre precisão e recall.

Trate isso como uma sugestão de partida, não como veredito. Se o seu projeto se importa mais em não
perder cantos, escolha um limiar mais baixo na curva e aceite o custo em precisão; se se importa
mais em não revisar falsos positivos, escolha um mais alto.

In [ ]:
#@title 🏆 Melhor ponto de operação { display-mode: "form" }

import pandas as pd

best_rows = []
for label in list(EVAL_LABELS) + [MICRO_AVERAGE_LABEL, MACRO_AVERAGE_LABEL]:
    scored = [r for r in metric_rows if r['label'] == label and r['f1'] is not None]
    if not scored:
        continue
    # Em caso de empate vence o limiar mais baixo: mesmo F1 com filtragem menos agressiva.
    best = min(scored, key=lambda r: (-r['f1'], r['score_threshold']))
    best_rows.append({
        'label': label,
        'score_threshold': best['score_threshold'],
        'sigmoid_bias': best['sigmoid_bias'],
        'precision': None if best['precision'] is None else round(best['precision'], 3),
        'recall': None if best['recall'] is None else round(best['recall'], 3),
        'f1': round(best['f1'], 3),
        'TP': best['TP'], 'FP': best['FP'], 'FN': best['FN'],
        'support': best['support'],
    })

BEST_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.best_operating_points.csv')
best_table = pd.DataFrame(best_rows)
best_table.to_csv(BEST_CSV_PATH, index=False)

print(f'Modelo         : {MODEL_NAME}')
print(f'Granularidade  : {GRANULARITY}  |  Correspondência: {MATCH_MODE}')
print(f'Gravações      : {len(validation_files)}')
print()
display(best_table)
print()
print(f'Métricas de todos os pontos de operação : {METRICS_CSV_PATH}')
print(f'Melhores pontos de operação             : {BEST_CSV_PATH}')
print(f'Figuras                                 : {FIGURES_DIR}')

---
### Pronto

Agora você tem, no seu Google Drive:

- `<modelo>.validation_metrics.csv` — cada rótulo em cada ponto de operação (TP/FP/FN, precisão,
  recall, F1)
- `<modelo>.best_operating_points.csv` — o limiar e o bias de maior F1 por rótulo
- `<modelo>_figures/` — uma figura de precisão/recall por rótulo

**Para testar outros pontos de operação**, edite o formulário *Variação de parâmetros e Validação* na Etapa 2 e
reexecute a partir da Etapa 6. Com o cache de logits ativado, a Etapa 5 não reexecuta o modelo.

---

Criado por [biodiversica](https://biodiversica.xyz). Para problemas, dúvidas ou sugestões, abra uma
issue no [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) ou visite
[biodiversica.xyz](https://biodiversica.xyz).